In [ ]:
# Modelo predictivo en Apache Spark MLlib
import pandas as pd
from pyspark.sql import SparkSession

archivo_excel = '03. Apoyo prueba - ventas_simuladas.xlsx'
archivo_csv = 'ventas_simuladas.csv'

df = pd.read_excel(archivo_excel)
df.to_csv(archivo_csv, index=False)
print("-> Excel convertido a CSV exitosamente.")

-> Excel convertido a CSV exitosamente.


In [13]:
from pyspark.sql import SparkSession
from datetime import datetime
import re
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("ModeloRiesgoMinimo").getOrCreate()
sc = spark.sparkContext

# CARGA Y LIMPIEZA DE DATOS
rdd_base = sc.textFile("ventas_simuladas.csv")
header = rdd_base.first()
rdd_base = rdd_base.filter(lambda linea: linea != header)

rdd_tuplas = rdd_base.map(lambda linea: tuple(linea.split(",")))

PATRON_FECHA_CORRUPTA = re.compile(r"^(\d{4})-(\d{2})-01 00:00:00$")

def es_fecha_corrupta(valor):
    return PATRON_FECHA_CORRUPTA.match(valor) is not None

def aprox_desde_fecha(valor):
    m = PATRON_FECHA_CORRUPTA.match(valor)
    anio, mes = int(m.group(1)), int(m.group(2))
    return anio + mes / 10.0

def valor_recuperado(valor):
    if es_fecha_corrupta(valor):
        return aprox_desde_fecha(valor)
    try:
        return float(valor)
    except (TypeError, ValueError):
        return None

def es_descartable(fila):
    if len(fila) != 6:
        return True
    sucursal, producto, cantidad, precio, monto, fecha_hora = fila
    if sucursal == "" or producto == "":
        return True
    if fecha_hora == "invalid-date":
        return True
    try:
        datetime.strptime(fecha_hora, "%Y-%m-%d %H:%M:%S")
    except ValueError:
        return True
    if cantidad == "":
        precio_f = valor_recuperado(precio)
        monto_f = valor_recuperado(monto)
        if precio_f is None or monto_f is None or precio_f == 0:
            return True
    return False

rdd_filtrado = rdd_tuplas.filter(lambda f: not es_descartable(f))

def transformar(fila):
    sucursal, producto, cantidad, precio, monto, fecha_hora = fila
    if cantidad == "":
        precio_corrupto = es_fecha_corrupta(precio)
        monto_corrupto = es_fecha_corrupta(monto)
        if not precio_corrupto and not monto_corrupto:
            precio_f = float(precio)
            monto_f = float(monto)
            cantidad = round(monto_f / precio_f)
        elif not precio_corrupto and monto_corrupto:
            precio_f = float(precio)
            monto_aprox = aprox_desde_fecha(monto)
            cantidad = round(monto_aprox / precio_f)
            monto_f = round(cantidad * precio_f, 2)
        elif precio_corrupto and not monto_corrupto:
            monto_f = float(monto)
            precio_aprox = aprox_desde_fecha(precio)
            cantidad = round(monto_f / precio_aprox)
            precio_f = round(monto_f / cantidad, 2)
        else:
            precio_f = round(aprox_desde_fecha(precio), 2)
            monto_aprox = aprox_desde_fecha(monto)
            cantidad = round(monto_aprox / precio_f)
            monto_f = round(cantidad * precio_f, 2)
    else:
        cantidad = int(float(cantidad))
        precio_corrupto = es_fecha_corrupta(precio)
        monto_corrupto = es_fecha_corrupta(monto)
        if not precio_corrupto and not monto_corrupto:
            precio_f = float(precio)
            monto_f = float(monto)
        elif not precio_corrupto and monto_corrupto:
            precio_f = float(precio)
            monto_f = round(cantidad * precio_f, 2)
        elif precio_corrupto and not monto_corrupto:
            monto_f = float(monto)
            precio_f = round(monto_f / cantidad, 2)
        else:
            precio_f = round(aprox_desde_fecha(precio), 2)
            monto_f = round(cantidad * precio_f, 2)

    fecha, hora = fecha_hora.split(" ")
    return (sucursal, producto, cantidad, precio_f, monto_f, fecha, hora)

rdd_limpio = rdd_filtrado.map(transformar)
rdd_limpio.cache()

df_ventas = rdd_limpio.map(lambda f: {
    "sucursal": f[0],
    "producto": f[1],
    "cantidad": int(f[2]),
    "precio": float(f[3]),
    "monto": float(f[4]),
    "fecha": f[5],
    "hora": f[6]
}).toDF()

# 1. PREPARAR EL DATASET PARA ENTRENAMIENTO DEL MODELO
df_etiquetado = df_ventas.withColumn(
    "label",
    F.when((F.col("monto") > 7000) | (F.col("hora") < "06:00:00"), 1.0).otherwise(0.0)
)

indexer_sucursal = StringIndexer(inputCol="sucursal", outputCol="sucursal_idx", handleInvalid="keep")
indexer_producto = StringIndexer(inputCol="producto", outputCol="producto_idx", handleInvalid="keep")

# Deje montofuera del vector de features porque es la misma variable que define la regla del label (monto > 7000), entonces incluirla generaría data leakage.
assembler = VectorAssembler(
    inputCols=["cantidad", "precio", "sucursal_idx", "producto_idx"],
    outputCol="features"
)

df_prep = Pipeline(stages=[indexer_sucursal, indexer_producto, assembler]).fit(df_etiquetado).transform(df_etiquetado)

print("=== VALIDACIÓN DE ESQUEMA ===")
df_prep.printSchema()

print("=== EVIDENCIA DATAFRAME FINAL (FEATURES Y LABEL) ===")
df_prep.select("features", "label").show(5, truncate=False)

# 2. ENTRENAR UN MODELO SUPERVISADO DE CLASIFICACIÓN CON MLLIB
train_data, test_data = df_etiquetado.randomSplit([0.7, 0.3], seed=42)

pipeline = Pipeline(stages=[
    indexer_sucursal,
    indexer_producto,
    assembler,
    RandomForestClassifier(labelCol="label", featuresCol="features", seed=42)
])

model = pipeline.fit(train_data)
predictions = model.transform(test_data)

print("=== TABLA DE PREDICCIONES ===")
predictions.select("label", "prediction", "probability").show(10, truncate=False)
print("")
# 3. EVALUAR EL MODELO Y JUSTIFICAR DECISIONES TÉCNICAS
accuracy = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
).evaluate(predictions)

auc = BinaryClassificationEvaluator(
    labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
).evaluate(predictions)

print(f"ACCURACY: {accuracy:.4f}")
print(f"AREAUNDERROC: {auc:.4f}")

# JUSTIFICACIÓN
# Elegí RandomForestClassifier porque tiene un buen manejo de variables mixtas y
# resistencia al sobreajuste. De igual forma tuve que excluir el monto de las
# features porque define directamente el label (monto > 7000) entonces asi se
# evita el data leakage y las métricas infladas. En cuanto a los resultados
# de Acurracy y de AreaUnderROC reflejan la capacidad real del modelo para detectar
# riesgo a partir de patrones de compra.
# En cuanto a los resultados, se obtuvo un Accuracy de 0.7812 y un AreaUnderROC
# de 0.8062. El AUC por sobre el accuracy indica que el modelo tiene una buena
# capacidad para discriminar entre transacciones riesgosas y normales en distintos
# umbrales de decisión, no solo acertando la clase mayoritaria. Lo que sugiere que
# el modelo captura patrones reales de compra asociados al riesgo (monto alto, horario de madrugada)
# y no está simplemente memorizando la regla del label a través de variables correlacionadas.
# Desde la perspectiva del negocio, esto es relevante porque un falso negativo
# (no detectar una venta riesgosa) tiene mayor costo operativo que un falso
# positivo, ya que implica dejar pasar errores de stock o transacciones atípicas
# sin revisión manual.
# Y como mejora futura, aplicaría CrossValidator para ajustar numTrees y maxDepth,
# y evaluaría también F1-score para balancear precisión y recall, dado el
# probable desbalance entre transacciones riesgosas y normales en el dataset.

spark.stop()

=== VALIDACIÓN DE ESQUEMA ===
root
 |-- cantidad: long (nullable = true)
 |-- fecha: string (nullable = true)
 |-- hora: string (nullable = true)
 |-- monto: double (nullable = true)
 |-- precio: double (nullable = true)
 |-- producto: string (nullable = true)
 |-- sucursal: string (nullable = true)
 |-- label: double (nullable = false)
 |-- sucursal_idx: double (nullable = false)
 |-- producto_idx: double (nullable = false)
 |-- features: vector (nullable = true)

=== EVIDENCIA DATAFRAME FINAL (FEATURES Y LABEL) ===
+---------------------+-----+
|features             |label|
+---------------------+-----+
|[3.0,989.78,0.0,1.0] |0.0  |
|[4.0,563.57,0.0,2.0] |0.0  |
|[4.0,1180.5,1.0,0.0] |1.0  |
|[3.0,2194.99,3.0,1.0]|0.0  |
|[5.0,1086.36,3.0,1.0]|1.0  |
+---------------------+-----+
only showing top 5 rows
=== TABLA DE PREDICCIONES ===
+-----+----------+----------------------------------------+
|label|prediction|probability                             |
+-----+----------+---------------